# Prueba del Taller 1 en un entorno distinto (Google Colab, runtime de R)

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución* y elige **R**.

**Preparación:** en tu computador, dentro de `parcial_01`, selecciona la carpeta `raw` y el archivo `scraper.R`, haz clic derecho → *Enviar a → Carpeta comprimida* y llámala `prueba_colab.zip`. Sube ese archivo a Colab con el icono de carpeta del panel izquierdo.

## 1. Descomprimir el proyecto

In [ ]:
unzip("prueba_colab.zip", exdir = "proyecto")
setwd("proyecto")
list.files()

## 2. Instalar los paquetes (puede tardar varios minutos)

In [ ]:
install.packages("pacman")
pacman::p_load(rvest, httr2, jsonlite, DBI, RSQLite, dplyr, stringr, purrr, tibble)
R.version.string

## 3. Modo sin conexión: reconstruir la base desde `raw/`
No consulta ningún sitio ni necesita Chrome. Cambia las fechas por las carpetas que tengas en `raw/<comercio>/`.

In [ ]:
source("scraper.R")

fechas <- c("2026-10-07", "2026-10-09")
for (f in fechas) {
  ejecutar_recoleccion_completa(".", ruta_db = "prueba_colab.sqlite", modo = "offline", fecha = f)
}

con <- dbConnect(RSQLite::SQLite(), "prueba_colab.sqlite")
dbGetQuery(con, "
  SELECT     c.nombre AS comercio,
             COUNT(DISTINCT a.id_anuncio) AS anuncios,
             COUNT(*)                     AS observaciones
  FROM       observaciones o
  INNER JOIN anuncios  a ON o.id_anuncio  = a.id_anuncio
  INNER JOIN comercios c ON a.id_comercio = c.id_comercio
  GROUP BY   c.nombre;
")

Compara esta tabla con la de tu computador: si coincide, el procesamiento es reproducible en otro entorno.

## 4. Modo en línea (solo La Rebaja y Locatel, sin Chrome)

In [ ]:
dir.create("raw_colab", showWarnings = FALSE)

lr  <- scrape_la_rebaja("colab_larebaja", "raw_colab/la_rebaja", "online")
loc <- scrape_locatel("colab_locatel", "raw_colab/locatel", "online")

c(filas_la_rebaja = nrow(lr),  total_la_rebaja = attr(lr, "total_sitio"),
  filas_locatel   = nrow(loc), total_locatel   = attr(loc, "total_sitio"))

## 5. (Opcional) Farmatodo, que necesita Chrome
Si esta parte falla, no es grave: se reporta como limitación del entorno.

In [ ]:
system("wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb")
system("apt-get install -y ./google-chrome-stable_current_amd64.deb > /dev/null 2>&1")
Sys.setenv(CHROMOTE_CHROME = "/usr/bin/google-chrome-stable")
chromote::set_chrome_args(c("--no-sandbox", "--disable-gpu", "--disable-dev-shm-usage"))

fa <- scrape_farmatodo("colab_farmatodo", "raw_colab/farmatodo", "online")
nrow(fa)